# Step 1 - Verb Pool Construction via VerbNet

This notebook currently implements **only Step 1** from the roadmap in [context/dataset_generation_instructions.md](../../context/dataset_generation_instructions.md).

Scope in this notebook:
- Build clean and corrupt verb pools from VerbNet classes
- Convert lemmas to passive-usable past participles
- Apply single-token filtering with leading-space tokenization
- Export a Step 1 artifact for downstream steps

Out of scope for now:
- Step 2+ (WordNet noun pools, semantic frames, pair expansion, filtering, targets)

In [1]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path

import nltk
from nltk.corpus import verbnet as vn
from transformers import AutoTokenizer

MODEL_NAME = "gpt2"
OUTPUT_STEP1 = Path("../dataset/semantic_meaningful/step1_verbnet_verb_pools.json")

# VerbNet 3.4 in NLTK does not always annotate Cause with explicit -animate.
# This stays VerbNet-only (no manual seed lists), with a metadata fallback rule if needed.
ALLOW_VERBNET_METADATA_FALLBACK_FOR_CORRUPT = True

nltk.download("verbnet", quiet=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

print(f"VerbNet classes: {len(vn.classids())}")
print(f"Tokenizer model: {MODEL_NAME}")
print(f"Step 1 output: {OUTPUT_STEP1.resolve()}")

VerbNet classes: 429
Tokenizer model: gpt2
Step 1 output: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\step1_verbnet_verb_pools.json


In [2]:
try:
    from lemminflect import getInflection
    HAS_LEMMINFLECT = True
except Exception:
    HAS_LEMMINFLECT = False


IRREGULAR_VBN = {
    "arise": "arisen",
    "awake": "awoken",
    "be": "been",
    "bear": "borne",
    "beat": "beaten",
    "become": "become",
    "begin": "begun",
    "bend": "bent",
    "bet": "bet",
    "bind": "bound",
    "bite": "bitten",
    "bleed": "bled",
    "blow": "blown",
    "break": "broken",
    "bring": "brought",
    "build": "built",
    "buy": "bought",
    "catch": "caught",
    "choose": "chosen",
    "come": "come",
    "cost": "cost",
    "cut": "cut",
    "do": "done",
    "draw": "drawn",
    "drink": "drunk",
    "drive": "driven",
    "eat": "eaten",
    "fall": "fallen",
    "feed": "fed",
    "feel": "felt",
    "fight": "fought",
    "find": "found",
    "fly": "flown",
    "forget": "forgotten",
    "freeze": "frozen",
    "get": "gotten",
    "give": "given",
    "go": "gone",
    "grow": "grown",
    "hang": "hung",
    "have": "had",
    "hear": "heard",
    "hide": "hidden",
    "hold": "held",
    "keep": "kept",
    "know": "known",
    "lead": "led",
    "leave": "left",
    "lose": "lost",
    "make": "made",
    "meet": "met",
    "pay": "paid",
    "put": "put",
    "read": "read",
    "ride": "ridden",
    "ring": "rung",
    "rise": "risen",
    "run": "run",
    "say": "said",
    "see": "seen",
    "sell": "sold",
    "send": "sent",
    "set": "set",
    "shake": "shaken",
    "shine": "shone",
    "shoot": "shot",
    "show": "shown",
    "shut": "shut",
    "sing": "sung",
    "sink": "sunk",
    "sit": "sat",
    "sleep": "slept",
    "speak": "spoken",
    "spend": "spent",
    "split": "split",
    "stand": "stood",
    "steal": "stolen",
    "swear": "sworn",
    "swim": "swum",
    "take": "taken",
    "teach": "taught",
    "tear": "torn",
    "tell": "told",
    "think": "thought",
    "throw": "thrown",
    "understand": "understood",
    "wear": "worn",
    "win": "won",
    "write": "written",
}


def tok_len(word: str) -> int:
    return len(tokenizer.encode(" " + word))


def group_by_tok(words: list[str]) -> dict[int, list[str]]:
    grouped: dict[int, list[str]] = defaultdict(list)
    for word in words:
        grouped[tok_len(word)].append(word)
    return {k: sorted(set(v)) for k, v in sorted(grouped.items())}


def normalize_member_name(name: str | None) -> str | None:
    if not name:
        return None
    norm = name.strip().lower().replace("_", " ")
    norm = re.sub(r"[^a-z\s-]", "", norm)
    if " " in norm or "-" in norm or not norm:
        return None
    return norm


def collect_selrestrs(node) -> list[tuple[str, str]]:
    if node is None:
        return []
    out: list[tuple[str, str]] = []
    for sel in node.findall("SELRESTR"):
        out.append((sel.attrib.get("Value", ""), sel.attrib.get("type", "").lower()))
    for nested in node.findall("SELRESTRS"):
        out.extend(collect_selrestrs(nested))
    return out


def class_role_restrictions(vn_class_xml) -> dict[str, list[tuple[str, str]]]:
    role_map: dict[str, list[tuple[str, str]]] = defaultdict(list)
    for role in vn_class_xml.findall("THEMROLES/THEMROLE"):
        role_name = (role.attrib.get("type") or "").lower()
        role_map[role_name].extend(collect_selrestrs(role.find("SELRESTRS")))
    return role_map


def has_restriction(
    role_map: dict[str, list[tuple[str, str]]],
    role_name: str,
    value: str,
    restr_type: str,
) -> bool:
    return any(v == value and t == restr_type for v, t in role_map.get(role_name, []))


def to_past_participle(lemma: str) -> str:
    if HAS_LEMMINFLECT:
        forms = getInflection(lemma, tag="VBN")
        if forms:
            return forms[0].lower()

    if lemma in IRREGULAR_VBN:
        return IRREGULAR_VBN[lemma]

    if lemma.endswith("e"):
        return lemma + "d"

    if len(lemma) >= 2 and lemma.endswith("y") and lemma[-2] not in "aeiou":
        return lemma[:-1] + "ied"

    if (
        len(lemma) >= 3
        and lemma[-1] not in "aeiouwxy"
        and lemma[-2] in "aeiou"
        and lemma[-3] not in "aeiou"
    ):
        return lemma + lemma[-1] + "ed"

    return lemma + "ed"


print(f"lemminflect available: {HAS_LEMMINFLECT}")

lemminflect available: False


In [3]:
PHYSICAL_TYPES = {
    "concrete",
    "solid",
    "substance",
    "location",
    "artifact",
    "machine",
    "vehicle",
    "body_part",
}


def is_clean_class(role_map: dict[str, list[tuple[str, str]]]) -> bool:
    return has_restriction(role_map, "agent", "+", "animate") or has_restriction(
        role_map, "agent", "+", "human"
    )


def is_corrupt_class_strict(role_map: dict[str, list[tuple[str, str]]]) -> bool:
    return has_restriction(role_map, "cause", "-", "animate")


def is_corrupt_class_fallback(role_map: dict[str, list[tuple[str, str]]]) -> bool:
    has_cause_role = "cause" in role_map

    patient_or_theme_physical = False
    for role_name in ("patient", "theme"):
        for value, typ in role_map.get(role_name, []):
            if (value == "-" and typ == "animate") or (value == "+" and typ in PHYSICAL_TYPES):
                patient_or_theme_physical = True

    instrument_physical = any(
        value == "+" and typ in {"concrete", "solid", "artifact"}
        for value, typ in role_map.get("instrument", [])
    )

    agent_intentional = any(
        value == "+" and typ in {"int_control", "animate", "human"}
        for value, typ in role_map.get("agent", [])
    )

    return patient_or_theme_physical and (has_cause_role or instrument_physical or agent_intentional)


clean_lemma_to_classes: dict[str, set[str]] = defaultdict(set)
corrupt_lemma_to_classes: dict[str, set[str]] = defaultdict(set)

strict_corrupt_classes: set[str] = set()
fallback_corrupt_classes: set[str] = set()

for class_id in vn.classids():
    vn_class = vn.vnclass(class_id)
    role_map = class_role_restrictions(vn_class)

    clean_match = is_clean_class(role_map)
    corrupt_match_strict = is_corrupt_class_strict(role_map)

    if corrupt_match_strict:
        strict_corrupt_classes.add(class_id)

    corrupt_match = corrupt_match_strict
    if not corrupt_match and ALLOW_VERBNET_METADATA_FALLBACK_FOR_CORRUPT:
        corrupt_match = is_corrupt_class_fallback(role_map)
        if corrupt_match:
            fallback_corrupt_classes.add(class_id)

    if not clean_match and not corrupt_match:
        continue

    member_lemmas = [
        normalize_member_name(member.attrib.get("name"))
        for member in vn_class.findall("MEMBERS/MEMBER")
    ]
    member_lemmas = [m for m in member_lemmas if m is not None]

    for lemma in member_lemmas:
        if clean_match:
            clean_lemma_to_classes[lemma].add(class_id)
        if corrupt_match:
            corrupt_lemma_to_classes[lemma].add(class_id)

print(f"Clean lemmas from VerbNet: {len(clean_lemma_to_classes)}")
print(f"Corrupt lemmas from VerbNet: {len(corrupt_lemma_to_classes)}")
print(f"Corrupt strict classes (Cause -animate): {len(strict_corrupt_classes)}")
print(f"Corrupt fallback classes (VerbNet metadata): {len(fallback_corrupt_classes)}")

Clean lemmas from VerbNet: 1794
Corrupt lemmas from VerbNet: 881
Corrupt strict classes (Cause -animate): 0
Corrupt fallback classes (VerbNet metadata): 53


In [4]:
def convert_pool(lemma_to_classes: dict[str, set[str]]) -> tuple[dict[str, set[str]], list[dict[str, str]]]:
    participle_to_classes: dict[str, set[str]] = defaultdict(set)
    dropped: list[dict[str, str]] = []

    for lemma, class_ids in lemma_to_classes.items():
        participle = to_past_participle(lemma)

        if not re.fullmatch(r"[a-z]+", participle):
            dropped.append({"lemma": lemma, "reason": "non_alpha_participle", "candidate": participle})
            continue

        participle_to_classes[participle].update(class_ids)

    return participle_to_classes, dropped


clean_participle_to_classes, clean_dropped = convert_pool(clean_lemma_to_classes)
corrupt_participle_to_classes, corrupt_dropped = convert_pool(corrupt_lemma_to_classes)

clean_participles = set(clean_participle_to_classes)
corrupt_participles = set(corrupt_participle_to_classes)

overlap = clean_participles.intersection(corrupt_participles)
for verb in overlap:
    clean_participle_to_classes.pop(verb, None)
    corrupt_participle_to_classes.pop(verb, None)

clean_participles = sorted(clean_participle_to_classes)
corrupt_participles = sorted(corrupt_participle_to_classes)

clean_by_tok = group_by_tok(clean_participles)
corrupt_by_tok = group_by_tok(corrupt_participles)

clean_1tok = clean_by_tok.get(1, [])
corrupt_1tok = corrupt_by_tok.get(1, [])

print(f"Clean participles after conversion: {len(clean_participles)}")
print(f"Corrupt participles after conversion: {len(corrupt_participles)}")
print(f"Overlap removed for disjointness: {len(overlap)}")
print(f"Clean 1-token verbs: {len(clean_1tok)}")
print(f"Corrupt 1-token verbs: {len(corrupt_1tok)}")

Clean participles after conversion: 1038
Corrupt participles after conversion: 124
Overlap removed for disjointness: 756
Clean 1-token verbs: 388
Corrupt 1-token verbs: 26


In [5]:
assert len(clean_1tok) > 0, "No 1-token clean verbs produced."
assert len(corrupt_1tok) > 0, "No 1-token corrupt verbs produced."
assert set(clean_1tok).isdisjoint(set(corrupt_1tok)), "Clean/corrupt 1-token pools overlap."

print("Sample clean verbs with provenance:")
for verb in clean_1tok[:15]:
    classes = sorted(clean_participle_to_classes[verb])[:3]
    print(f"  {verb:<16} {classes}")

print("\nSample corrupt verbs with provenance:")
for verb in corrupt_1tok[:15]:
    classes = sorted(corrupt_participle_to_classes[verb])[:3]
    print(f"  {verb:<16} {classes}")

print("\nStep 1 validation checks passed.")

Sample clean verbs with provenance:
  abducted         ['steal-10.5']
  abused           ['judgement-33']
  accepted         ['approve-77', 'obtain-13.5.2']
  acclaimed        ['judgement-33']
  accompanied      ['accompany-51.7']
  accrued          ['obtain-13.5.2']
  accused          ['suspect-81']
  acknowledged     ['confess-37.10']
  acquired         ['learn-14']
  addressed        ['illustrate-25.3']
  admitted         ['admit-65', 'confess-37.10']
  advanced         ['future_having-13.3']
  advertised       ['search-35.2']
  affiliated       ['amalgamate-22.2-2']
  alerted          ['advise-37.9']

Sample corrupt verbs with provenance:
  burned           ['tingle-40.8.2']
  cracked          ['break-45.1']
  crashed          ['break-45.1']
  crushed          ['break-45.1']
  damaged          ['destroy-44']
  demolished       ['destroy-44']
  destroyed        ['destroy-44']
  devastated       ['destroy-44']
  discarded        ['throw-17.1']
  distilled        ['wipe_manner-10.4.1'

In [6]:
artifact = {
    "meta": {
        "phase": "Step 1 only",
        "source": "VerbNet via nltk.corpus.verbnet",
        "tokenizer_model": MODEL_NAME,
        "token_length_uses_leading_space": True,
        "corrupt_primary_rule": "Cause role with -animate restriction",
        "corrupt_fallback_rule": (
            "VerbNet metadata fallback: physical Patient/Theme with Cause/Instrument/intentional-Agent signature"
            if ALLOW_VERBNET_METADATA_FALLBACK_FOR_CORRUPT
            else "disabled"
        ),
        "strict_corrupt_class_count": len(strict_corrupt_classes),
        "fallback_corrupt_class_count": len(fallback_corrupt_classes),
        "overlap_removed_count": len(overlap),
        "clean_dropped_count": len(clean_dropped),
        "corrupt_dropped_count": len(corrupt_dropped),
    },
    "verbs": {
        "note": "Step 1 output only: clean/corrupt verb pools in past participle form.",
        "by_token_length": {
            str(n): {
                "clean": clean_by_tok.get(n, []),
                "corrupt": corrupt_by_tok.get(n, []),
                "n_pairs": len(clean_by_tok.get(n, [])) * len(corrupt_by_tok.get(n, [])),
            }
            for n in sorted(set(clean_by_tok) | set(corrupt_by_tok))
        },
    },
    "provenance": {
        "clean_classes_by_verb": {
            verb: sorted(class_ids) for verb, class_ids in sorted(clean_participle_to_classes.items())
        },
        "corrupt_classes_by_verb": {
            verb: sorted(class_ids) for verb, class_ids in sorted(corrupt_participle_to_classes.items())
        },
        "strict_corrupt_classes": sorted(strict_corrupt_classes),
        "fallback_corrupt_classes": sorted(fallback_corrupt_classes),
        "dropped": {
            "clean": clean_dropped,
            "corrupt": corrupt_dropped,
        },
    },
}

OUTPUT_STEP1.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_STEP1, "w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

print(f"Saved Step 1 artifact to: {OUTPUT_STEP1.resolve()}")
print(f"1-token clean pool size: {len(clean_1tok)}")
print(f"1-token corrupt pool size: {len(corrupt_1tok)}")

Saved Step 1 artifact to: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\step1_verbnet_verb_pools.json
1-token clean pool size: 388
1-token corrupt pool size: 26


## Step 2 - Noun Pool Construction via WordNet

This section implements Step 2 from the roadmap:

- Build animate nouns from WordNet hyponyms of `person.n.01`
- Build inanimate agent nouns from WordNet hyponyms of `natural_phenomenon.n.01` plus physical `artifact.n.01` descendants
- Normalize to singular forms with `inflect`
- Apply single-token filtering with leading-space tokenization
- Verify animate/inanimate pool disjointness
- Export Step 2 artifacts for downstream steps

Run the new code cells below in order after reviewing them.

In [7]:
from nltk.corpus import wordnet as wn
from transformers import AutoTokenizer

import inflect

OUTPUT_STEP2 = Path("../dataset/semantic_meaningful/step2_wordnet_noun_pools.json")
OUTPUT_CURATED = Path("../dataset/semantic_meaningful/curated_vocabulary.json")

if "MODEL_NAME" not in globals():
    MODEL_NAME = "gpt2"

if "OUTPUT_STEP1" not in globals():
    OUTPUT_STEP1 = Path("../dataset/semantic_meaningful/step1_verbnet_verb_pools.json")

assert OUTPUT_STEP1.exists(), f"Missing Step 1 artifact: {OUTPUT_STEP1.resolve()}"
with open(OUTPUT_STEP1, "r", encoding="utf-8") as f:
    step1_artifact = json.load(f)

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

infl = inflect.engine()

print(f"Tokenizer model: {MODEL_NAME}")
print(f"Loaded Step 1 artifact: {OUTPUT_STEP1.resolve()}")
print(f"Step 2 output: {OUTPUT_STEP2.resolve()}")
print(f"Curated vocab output: {OUTPUT_CURATED.resolve()}")

Tokenizer model: gpt2
Loaded Step 1 artifact: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\step1_verbnet_verb_pools.json
Step 2 output: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\step2_wordnet_noun_pools.json
Curated vocab output: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\curated_vocabulary.json


In [8]:
ABSTRACT_ARTIFACT_KEYWORDS = {
    "software",
    "program",
    "document",
    "record",
    "symbol",
    "idea",
    "concept",
    "theory",
    "plan",
    "method",
    "formula",
    "message",
    "language",
    "code",
}

ARTIFACT_PHYSICAL_ANCHORS = [
    "vehicle.n.01",
    "structure.n.01",
    "tool.n.01",
    "instrumentality.n.03",
    "container.n.01",
    "weapon.n.01",
]

if "group_by_tok" not in globals():
    def group_by_tok(words: list[str]) -> dict[int, list[str]]:
        grouped: dict[int, list[str]] = defaultdict(list)
        for word in words:
            grouped[len(tokenizer.encode(" " + word))].append(word)
        return {k: sorted(set(v)) for k, v in sorted(grouped.items())}


def all_hyponyms(root_synset):
    seen = set()
    stack = [root_synset]
    while stack:
        syn = stack.pop()
        if syn in seen:
            continue
        seen.add(syn)
        stack.extend(syn.hyponyms())
    return seen


def normalize_lemma_name(lemma_name: str) -> str | None:
    token = lemma_name.lower().replace("_", " ").strip()
    if " " in token or "-" in token:
        return None
    if not re.fullmatch(r"[a-z]+", token):
        return None
    return token


def singularize_word(word: str) -> str | None:
    singular = infl.singular_noun(word)
    candidate = singular if isinstance(singular, str) and singular else word
    if not re.fullmatch(r"[a-z]+", candidate):
        return None
    return candidate.lower()


def is_concrete_artifact_synset(synset) -> bool:
    definition = synset.definition().lower()
    return not any(keyword in definition for keyword in ABSTRACT_ARTIFACT_KEYWORDS)


def words_from_synsets(synsets, concrete_artifact_filter: bool = False) -> set[str]:
    words: set[str] = set()
    for syn in synsets:
        if concrete_artifact_filter and not is_concrete_artifact_synset(syn):
            continue
        for lemma in syn.lemma_names():
            norm = normalize_lemma_name(lemma)
            if norm is None:
                continue
            sg = singularize_word(norm)
            if sg is None:
                continue
            words.add(sg)
    return words


person_synsets = all_hyponyms(wn.synset("person.n.01"))
natural_synsets = all_hyponyms(wn.synset("natural_phenomenon.n.01"))

artifact_synsets = set()
for anchor_name in ARTIFACT_PHYSICAL_ANCHORS:
    artifact_synsets |= all_hyponyms(wn.synset(anchor_name))

animate_all = words_from_synsets(person_synsets)
inanimate_natural_all = words_from_synsets(natural_synsets)
inanimate_artifact_all = words_from_synsets(artifact_synsets, concrete_artifact_filter=True)
inanimate_all = inanimate_natural_all.union(inanimate_artifact_all)

overlap_all = animate_all.intersection(inanimate_all)
if overlap_all:
    inanimate_all = inanimate_all - overlap_all

assert animate_all.isdisjoint(inanimate_all), "Animate/inanimate pools overlap after disjointness filtering."

animate_by_tok = group_by_tok(sorted(animate_all))
inanimate_by_tok = group_by_tok(sorted(inanimate_all))

animate_1tok = sorted(animate_by_tok.get(1, []))
inanimate_1tok = sorted(inanimate_by_tok.get(1, []))

assert set(animate_1tok).isdisjoint(set(inanimate_1tok)), "Animate/inanimate 1-token noun pools overlap."
assert len(animate_1tok) > 0, "No 1-token animate nouns produced."
assert len(inanimate_1tok) > 0, "No 1-token inanimate nouns produced."

print(f"Animate noun candidates (all token lengths): {len(animate_all)}")
print(f"Inanimate natural candidates (all token lengths): {len(inanimate_natural_all)}")
print(f"Inanimate artifact candidates (all token lengths): {len(inanimate_artifact_all)}")
print(f"Inanimate noun candidates (union, post-disjoint): {len(inanimate_all)}")
print(f"Full-set overlap removed from inanimate pool: {len(overlap_all)}")
print(f"Animate 1-token nouns: {len(animate_1tok)}")
print(f"Inanimate 1-token nouns: {len(inanimate_1tok)}")

print("\nSample animate 1-token nouns:", animate_1tok[:20])
print("\nSample inanimate 1-token nouns:", inanimate_1tok[:20])

Animate noun candidates (all token lengths): 7443
Inanimate natural candidates (all token lengths): 478
Inanimate artifact candidates (all token lengths): 4477
Inanimate noun candidates (union, post-disjoint): 4540
Full-set overlap removed from inanimate pool: 382
Animate 1-token nouns: 1285
Inanimate 1-token nouns: 1225

Sample animate 1-token nouns: ['aboriginal', 'absentee', 'abuser', 'academic', 'accessory', 'accountant', 'accused', 'accuser', 'ace', 'acquaintance', 'active', 'activist', 'actor', 'adapter', 'addict', 'adept', 'adherent', 'adjunct', 'administrator', 'adolescent']

Sample inanimate 1-token nouns: ['ac', 'academy', 'accelerator', 'accommodation', 'action', 'aerial', 'affinity', 'air', 'aircraft', 'airplane', 'aisle', 'alarm', 'altar', 'ambulance', 'ammo', 'ammunition', 'amplifier', 'android', 'antenna', 'apartment']


In [9]:
def serialize_pool(by_tok: dict[int, list[str]], note: str) -> dict:
    return {
        "note": note,
        "by_token_length": {str(k): {"words": v} for k, v in sorted(by_tok.items())},
    }


animate_pool = serialize_pool(
    animate_by_tok,
    "Animate noun pool from WordNet hyponyms of person.n.01 (singularized).",
)
inanimate_pool = serialize_pool(
    inanimate_by_tok,
    "Inanimate agent pool from natural_phenomenon.n.01 and physical artifact descendants (singularized).",
)

# Ensure the exported 1-token pools are exactly the disjoint versions used downstream.
animate_pool["by_token_length"]["1"] = {"words": animate_1tok}
inanimate_pool["by_token_length"]["1"] = {"words": inanimate_1tok}

step2_artifact = {
    "meta": {
        "phase": "Step 2 only",
        "source": "WordNet via nltk.corpus.wordnet",
        "tokenizer_model": MODEL_NAME,
        "token_length_uses_leading_space": True,
        "animate_root": "person.n.01",
        "inanimate_roots": ["natural_phenomenon.n.01", "artifact physical anchors"],
        "artifact_anchors": ARTIFACT_PHYSICAL_ANCHORS,
        "inflect_singularization": True,
        "removed_overlap_from_inanimate_count": len(overlap_all),
    },
    "nouns": {
        "animate_pool": animate_pool,
        "inanimate_pool": inanimate_pool,
    },
}

OUTPUT_STEP2.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_STEP2, "w", encoding="utf-8") as f:
    json.dump(step2_artifact, f, indent=2)

curated_vocabulary = {
    "meta": {
        "phase": "Step 1 + Step 2",
        "description": "VerbNet verb pools plus WordNet noun pools for animacy s-selection dataset generation.",
        "tokenizer_model": MODEL_NAME,
        "token_length_uses_leading_space": True,
        "number": "singular - all nouns stored in singular form",
    },
    "verbs": step1_artifact["verbs"],
    "animate_pool": animate_pool,
    "inanimate_pool": inanimate_pool,
    "human_pool": animate_pool,
    "force_pool": inanimate_pool,
}

OUTPUT_CURATED.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_CURATED, "w", encoding="utf-8") as f:
    json.dump(curated_vocabulary, f, indent=2)

print(f"Saved Step 2 artifact to: {OUTPUT_STEP2.resolve()}")
print(f"Updated curated vocabulary: {OUTPUT_CURATED.resolve()}")
print(f"Animate 1-token pool size: {len(animate_1tok)}")
print(f"Inanimate 1-token pool size: {len(inanimate_1tok)}")
print("Step 2 build is ready. Execute this cell to write outputs.")

Saved Step 2 artifact to: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\step2_wordnet_noun_pools.json
Updated curated vocabulary: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\curated_vocabulary.json
Animate 1-token pool size: 1285
Inanimate 1-token pool size: 1225
Step 2 build is ready. Execute this cell to write outputs.


## Step 3 - Verb Pair Definition and Semantic Domain Grouping

This step derives `(clean_verb, corrupt_verb)` pairs directly from VerbNet provenance produced in Step 1, then groups pairs into automatically-derived domains.

Design choices in this implementation:
- Domain IDs are derived from VerbNet superclass family numbers (no manual domain names).
- Pair sharpness uses two automatic signals: VerbNet family gap and lexical dissimilarity.
- A cap is applied per domain to prevent domain imbalance in later expansion steps.

In [ ]:
from collections import Counter
from difflib import SequenceMatcher
from itertools import product

OUTPUT_STEP3 = Path("../dataset/semantic_meaningful/step3_verb_pairs_by_domain.json")

# Keep Step 3 tractable for Step 4 LLM assignment while preserving domain diversity.
MAX_DOMAINS = 40
MAX_TOTAL_PAIRS = 320
MAX_PAIRS_PER_DOMAIN = 8
MIN_PAIRS_PER_DOMAIN = 4

# Sharper contrast than the first pass.
MIN_FAMILY_GAP = 6
MAX_LEXICAL_SIMILARITY = 0.4

clean_classes_by_verb = step1_artifact["provenance"]["clean_classes_by_verb"]
corrupt_classes_by_verb = step1_artifact["provenance"]["corrupt_classes_by_verb"]


def canonical_vn_class(class_id: str) -> str:
    # Keep only the canonical VerbNet class stem, removing subclass suffixes like "-1".
    m = re.match(r"^(.+?-\d+(?:\.\d+)*)", class_id)
    return m.group(1) if m else class_id


def class_family(class_id: str) -> int | None:
    canonical = canonical_vn_class(class_id)
    if "-" not in canonical:
        return None
    numeric = canonical.rsplit("-", 1)[-1]
    head = numeric.split(".")[0]
    return int(head) if head.isdigit() else None


def representative_class(class_ids: list[str]) -> str | None:
    if not class_ids:
        return None
    canonical = [canonical_vn_class(c) for c in class_ids]
    return Counter(canonical).most_common(1)[0][0]


def lexical_similarity(a: str, b: str) -> float:
    return SequenceMatcher(a=a, b=b).ratio()


clean_rep_class = {v: representative_class(clean_classes_by_verb.get(v, [])) for v in clean_1tok}
corrupt_rep_class = {v: representative_class(corrupt_classes_by_verb.get(v, [])) for v in corrupt_1tok}

clean_family = {v: class_family(cls) if cls else None for v, cls in clean_rep_class.items()}
corrupt_family = {v: class_family(cls) if cls else None for v, cls in corrupt_rep_class.items()}

candidate_rows = []
for clean_v, corrupt_v in product(clean_1tok, corrupt_1tok):
    fam_c = clean_family.get(clean_v)
    fam_i = corrupt_family.get(corrupt_v)
    if fam_c is None or fam_i is None:
        continue

    fam_gap = abs(fam_c - fam_i)
    lex_sim = lexical_similarity(clean_v, corrupt_v)

    # "Sharp" contrast: families sufficiently separated and forms not too lexically similar.
    if fam_gap < MIN_FAMILY_GAP or lex_sim > MAX_LEXICAL_SIMILARITY:
        continue

    fam_lo, fam_hi = sorted((fam_c, fam_i))
    domain_id = f"vn_family_{fam_lo}_{fam_hi}"

    candidate_rows.append(
        {
            "domain": domain_id,
            "clean_verb": clean_v,
            "corrupt_verb": corrupt_v,
            "clean_class": clean_rep_class[clean_v],
            "corrupt_class": corrupt_rep_class[corrupt_v],
            "clean_family": fam_c,
            "corrupt_family": fam_i,
            "family_gap": fam_gap,
            "lexical_similarity": round(lex_sim, 4),
        }
    )

domain_rows: dict[str, list[dict]] = defaultdict(list)
for row in candidate_rows:
    domain_rows[row["domain"]].append(row)

for domain, rows in domain_rows.items():
    rows.sort(key=lambda r: (-r["family_gap"], r["lexical_similarity"], r["clean_verb"], r["corrupt_verb"]))

ranked_domains = []
for domain, rows in domain_rows.items():
    kept = rows[:MAX_PAIRS_PER_DOMAIN]
    if len(kept) < MIN_PAIRS_PER_DOMAIN:
        continue

    avg_gap = sum(r["family_gap"] for r in kept) / len(kept)
    avg_lex = sum(r["lexical_similarity"] for r in kept) / len(kept)
    ranked_domains.append((domain, avg_gap, avg_lex, len(kept), kept))

ranked_domains.sort(key=lambda x: (-x[1], x[2], -x[3], x[0]))

domain_pairs: dict[str, list[list[str]]] = {}
domain_details: dict[str, list[dict]] = {}
selected_domains = 0
selected_pairs = 0

for domain, avg_gap, avg_lex, n_kept, kept in ranked_domains:
    if selected_domains >= MAX_DOMAINS or selected_pairs >= MAX_TOTAL_PAIRS:
        break

    remaining = MAX_TOTAL_PAIRS - selected_pairs
    if remaining <= 0:
        break

    slice_size = min(n_kept, remaining)
    kept_slice = kept[:slice_size]
    if len(kept_slice) < MIN_PAIRS_PER_DOMAIN:
        continue

    domain_pairs[domain] = [[r["clean_verb"], r["corrupt_verb"]] for r in kept_slice]
    domain_details[domain] = kept_slice

    selected_domains += 1
    selected_pairs += len(kept_slice)

total_pairs = sum(len(v) for v in domain_pairs.values())
print(f"Candidate pairs after sharpness filter: {len(candidate_rows)}")
print(f"Domains retained after ranking/caps: {len(domain_pairs)}")
print(f"Total retained pairs after ranking/caps: {total_pairs}")

top_domains = sorted(domain_pairs.items(), key=lambda kv: len(kv[1]), reverse=True)[:10]
print("\nTop domains by retained pairs:")
for domain, pairs in top_domains:
    print(f"  {domain:<22} {len(pairs)} pairs")

Candidate pairs after sharpness filter: 4755
Domains retained after ranking/caps: 40
Total retained pairs after ranking/caps: 256

Top domains by retained pairs:
  vn_family_23_77        8 pairs
  vn_family_10_59        8 pairs
  vn_family_11_59        8 pairs
  vn_family_10_54        8 pairs
  vn_family_23_67        8 pairs
  vn_family_11_55        8 pairs
  vn_family_11_54        8 pairs
  vn_family_23_65        8 pairs
  vn_family_44_86        8 pairs
  vn_family_17_59        8 pairs


In [13]:
assert len(domain_pairs) > 0, "No Step 3 domains were produced. Relax filters or inspect Step 1 pools."
assert all(len(pairs) <= MAX_PAIRS_PER_DOMAIN for pairs in domain_pairs.values())
assert sum(len(v) for v in domain_pairs.values()) <= MAX_TOTAL_PAIRS

step3_artifact = {
    "meta": {
        "phase": "Step 3 only",
        "source": "Derived from Step 1 VerbNet provenance",
        "tokenizer_model": MODEL_NAME,
        "derivation": {
            "domain_id": "Pair of VerbNet family numbers from representative clean/corrupt classes",
            "sharpness_min_family_gap": MIN_FAMILY_GAP,
            "sharpness_max_lexical_similarity": MAX_LEXICAL_SIMILARITY,
            "max_domains": MAX_DOMAINS,
            "max_total_pairs": MAX_TOTAL_PAIRS,
            "max_pairs_per_domain": MAX_PAIRS_PER_DOMAIN,
            "min_pairs_per_domain": MIN_PAIRS_PER_DOMAIN,
        },
        "n_domains": len(domain_pairs),
        "n_pairs": sum(len(v) for v in domain_pairs.values()),
    },
    "domains": domain_pairs,
    "details": domain_details,
}

OUTPUT_STEP3.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_STEP3, "w", encoding="utf-8") as f:
    json.dump(step3_artifact, f, indent=2)

print(f"Saved Step 3 artifact to: {OUTPUT_STEP3.resolve()}")
print(f"Domain count: {step3_artifact['meta']['n_domains']}")
print(f"Pair count: {step3_artifact['meta']['n_pairs']}")

Saved Step 3 artifact to: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\step3_verb_pairs_by_domain.json
Domain count: 40
Pair count: 256


## Step 4 - LLM Semantic Frame Assignment (Per Verb Pair)

For each `(clean_verb, corrupt_verb)` pair from Step 3, select compatible noun subsets from Step 2 pools.

Implementation notes:
- Uses OpenAI Chat Completions with JSON output.
- Uses a JSONL cache for resume support.
- Applies strict post-hoc filtering so outputs contain only nouns from the provided pools.

In [14]:
import os
import time

OUTPUT_FRAMES = Path("../dataset/semantic_meaningful/semantic_frames.json")
OUTPUT_STEP4_CACHE = Path("../dataset/semantic_meaningful/step4_frame_cache.jsonl")
FRAME_ASSIGN_MODEL = "gpt-4o"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

with open(OUTPUT_STEP3, "r", encoding="utf-8") as f:
    step3_data = json.load(f)

animate_pool_1tok = curated_vocabulary["animate_pool"]["by_token_length"]["1"]["words"]
inanimate_pool_1tok = curated_vocabulary["inanimate_pool"]["by_token_length"]["1"]["words"]

pairs_for_step4 = []
for domain, pairs in step3_data["domains"].items():
    for clean_v, corrupt_v in pairs:
        pairs_for_step4.append(
            {
                "domain": domain,
                "clean_verb": clean_v,
                "corrupt_verb": corrupt_v,
                "pair_key": f"{domain}::{clean_v}::{corrupt_v}",
            }
        )

print(f"Step 4 model: {FRAME_ASSIGN_MODEL}")
print(f"Step 4 pairs: {len(pairs_for_step4)}")
print(f"Animate noun pool (1tok): {len(animate_pool_1tok)}")
print(f"Inanimate noun pool (1tok): {len(inanimate_pool_1tok)}")
print(f"Cache file: {OUTPUT_STEP4_CACHE.resolve()}")
print(f"API key present: {'yes' if OPENAI_API_KEY else 'no'}")

Step 4 model: gpt-4o
Step 4 pairs: 256
Animate noun pool (1tok): 1285
Inanimate noun pool (1tok): 1225
Cache file: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\step4_frame_cache.jsonl
API key present: yes


In [16]:
def build_step4_prompt(clean_verb: str, corrupt_verb: str, animate_pool: list[str], inanimate_pool: list[str]) -> str:
    return f"""You are a linguist building a psycholinguistics dataset.
Template: "The [PATIENT] was [VERB] by the [AGENT]."

Verb pair:
- animate verb (clean): {clean_verb}
- inanimate verb (corrupt): {corrupt_verb}

Available human nouns (for PATIENT and animate AGENT slots):
{json.dumps(animate_pool)}

Available inanimate nouns (for inanimate AGENT slot):
{json.dumps(inanimate_pool)}

Select subsets that make BOTH sentences individually plausible:
- patients: nouns plausible as PATIENT for BOTH verbs
- animate_agents: nouns plausible as AGENT for the animate verb
- inanimate_agents: nouns plausible as AGENT for the inanimate verb

Constraints:
- Return ONLY words that appear verbatim in the lists above
- Return 5 to 12 items per key where possible
- Return valid JSON only, no commentary

{{
  "patients": [...],
  "animate_agents": [...],
  "inanimate_agents": [...]
}}"""


def load_step4_cache(cache_path: Path) -> dict[str, dict]:
    cache = {}
    if not cache_path.exists():
        return cache
    with open(cache_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            cache[rec["pair_key"]] = rec
    return cache


def append_step4_cache(cache_path: Path, rec: dict) -> None:
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def validate_list(values, pool_set: set[str]) -> tuple[list[str], int]:
    kept = []
    dropped = 0
    for value in values if isinstance(values, list) else []:
        if isinstance(value, str) and value in pool_set:
            kept.append(value)
        else:
            dropped += 1
    # Deduplicate while preserving order.
    deduped = list(dict.fromkeys(kept))
    return deduped, dropped


step4_cache = load_step4_cache(OUTPUT_STEP4_CACHE)
print(f"Loaded Step 4 cache entries: {len(step4_cache)}")

if not OPENAI_API_KEY:
    print("OPENAI_API_KEY missing. Set it, then re-run this cell to execute Step 4 calls.")
else:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    animate_set = set(animate_pool_1tok)
    inanimate_set = set(inanimate_pool_1tok)

    completed = 0
    for i, pair in enumerate(pairs_for_step4, start=1):
        key = pair["pair_key"]
        if key in step4_cache:
            continue

        prompt = build_step4_prompt(
            pair["clean_verb"],
            pair["corrupt_verb"],
            animate_pool_1tok,
            inanimate_pool_1tok,
        )

        try:
            response = client.chat.completions.create(
                model=FRAME_ASSIGN_MODEL,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
            )
            payload = json.loads(response.choices[0].message.content)

            patients, dropped_p = validate_list(payload.get("patients", []), animate_set)
            animate_agents, dropped_a = validate_list(payload.get("animate_agents", []), animate_set)
            inanimate_agents, dropped_i = validate_list(payload.get("inanimate_agents", []), inanimate_set)

            rec = {
                "pair_key": key,
                "domain": pair["domain"],
                "clean_verb": pair["clean_verb"],
                "corrupt_verb": pair["corrupt_verb"],
                "patients": patients,
                "animate_agents": animate_agents,
                "inanimate_agents": inanimate_agents,
                "dropped_not_in_pool": {
                    "patients": dropped_p,
                    "animate_agents": dropped_a,
                    "inanimate_agents": dropped_i,
                },
                "model": FRAME_ASSIGN_MODEL,
                "ts": time.time(),
            }

            append_step4_cache(OUTPUT_STEP4_CACHE, rec)
            step4_cache[key] = rec
            completed += 1

            if completed % 10 == 0:
                print(f"Processed {completed} new pairs (cache size: {len(step4_cache)})")

            time.sleep(0.2)
        except Exception as e:
            print(f"Step 4 call failed at pair {i}/{len(pairs_for_step4)} ({key}): {e}")
            break

    print(f"Step 4 cache size after run: {len(step4_cache)}")

Loaded Step 4 cache entries: 0
Processed 10 new pairs (cache size: 10)
Processed 20 new pairs (cache size: 20)
Processed 30 new pairs (cache size: 30)
Processed 40 new pairs (cache size: 40)
Processed 50 new pairs (cache size: 50)
Processed 60 new pairs (cache size: 60)
Processed 70 new pairs (cache size: 70)
Processed 80 new pairs (cache size: 80)
Processed 90 new pairs (cache size: 90)
Processed 100 new pairs (cache size: 100)
Processed 110 new pairs (cache size: 110)
Processed 120 new pairs (cache size: 120)
Processed 130 new pairs (cache size: 130)
Processed 140 new pairs (cache size: 140)
Processed 150 new pairs (cache size: 150)
Processed 160 new pairs (cache size: 160)
Processed 170 new pairs (cache size: 170)
Processed 180 new pairs (cache size: 180)
Processed 190 new pairs (cache size: 190)
Processed 200 new pairs (cache size: 200)
Step 4 call failed at pair 204/256 (vn_family_45_85::defended::ripped): Error code: 429 - {'error': {'message': 'You exceeded your current quota, p

In [17]:
frames = []
for pair in pairs_for_step4:
    rec = step4_cache.get(pair["pair_key"])
    if not rec:
        continue
    frames.append(
        {
            "domain": rec["domain"],
            "clean_verb": rec["clean_verb"],
            "corrupt_verb": rec["corrupt_verb"],
            "patients": rec["patients"],
            "animate_agents": rec["animate_agents"],
            "inanimate_agents": rec["inanimate_agents"],
            "dropped_not_in_pool": rec.get("dropped_not_in_pool", {}),
        }
    )

semantic_frames = {
    "meta": {
        "phase": "Step 4 only",
        "description": "Per-verb-pair semantic frame assignments from Step 3 pairs",
        "source_step3": str(OUTPUT_STEP3),
        "model": FRAME_ASSIGN_MODEL,
        "n_pairs_requested": len(pairs_for_step4),
        "n_pairs_cached": len(step4_cache),
        "n_frames_exported": len(frames),
        "cache_path": str(OUTPUT_STEP4_CACHE),
    },
    "frames": frames,
}

OUTPUT_FRAMES.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_FRAMES, "w", encoding="utf-8") as f:
    json.dump(semantic_frames, f, indent=2)

print(f"Saved semantic frames to: {OUTPUT_FRAMES.resolve()}")
print(f"Frames exported: {len(frames)} / {len(pairs_for_step4)}")

Saved semantic frames to: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\semantic_frames.json
Frames exported: 203 / 256


## Step 5 - Combinatorial Expansion

Expand each semantic frame into minimal pairs via Cartesian product:
`patients × animate_agents × inanimate_agents`, with Step 5 constraints enforced.

Implemented checks:
- Skip `patient == animate_agent`
- Enforce clean/corrupt token-length parity
- Cap rows per `(clean_verb, corrupt_verb)` at 1000
- Shuffle final rows before writing `raw_pairs.jsonl`

In [ ]:
import random
import uuid
from itertools import product

OUTPUT_RAW_PAIRS = Path("../dataset/semantic_meaningful/raw_pairs.jsonl")
MAX_ROWS_PER_VERB_PAIR = 1000
MIN_ITEMS_PER_FRAME_KEY = 3
RANDOM_SEED = 42

valid_frames = [
    fr
    for fr in frames
    if len(fr.get("patients", [])) >= MIN_ITEMS_PER_FRAME_KEY
    and len(fr.get("animate_agents", [])) >= MIN_ITEMS_PER_FRAME_KEY
    and len(fr.get("inanimate_agents", [])) >= MIN_ITEMS_PER_FRAME_KEY
]

rows = []
pair_counts: dict[tuple[str, str], int] = defaultdict(int)
parity_failures = 0
collision_skips = 0

for fr in valid_frames:
    clean_verb = fr["clean_verb"]
    corrupt_verb = fr["corrupt_verb"]
    pair_key = (clean_verb, corrupt_verb)

    combos = [
        (p, aa, ia)
        for p, aa, ia in product(fr["patients"], fr["animate_agents"], fr["inanimate_agents"])
        if p != aa
    ]
    collision_skips += (len(fr["patients"]) * len(fr["animate_agents"]) * len(fr["inanimate_agents"]) - len(combos))

    remaining = MAX_ROWS_PER_VERB_PAIR - pair_counts[pair_key]
    if remaining <= 0:
        continue

    for patient, animate_agent, inanimate_agent in combos[:remaining]:
        clean_prefix = f"The {patient} was {clean_verb} by the"
        corrupt_prefix = f"The {patient} was {corrupt_verb} by the"

        if len(tokenizer.encode(clean_prefix)) != len(tokenizer.encode(corrupt_prefix)):
            parity_failures += 1
            continue

        rows.append(
            {
                "clean": clean_prefix,
                "corrupt": corrupt_prefix,
                "patient": patient,
                "clean_verb": clean_verb,
                "corrupt_verb": corrupt_verb,
                "animate_agent": animate_agent,
                "inanimate_agent": inanimate_agent,
                "domain": fr["domain"],
                "uid": uuid.uuid4().hex,
            }
        )
        pair_counts[pair_key] += 1

rng = random.Random(RANDOM_SEED)
rng.shuffle(rows)

OUTPUT_RAW_PAIRS.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_RAW_PAIRS, "w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Frames available: {len(frames)}")
print(f"Frames used (>= {MIN_ITEMS_PER_FRAME_KEY} per key): {len(valid_frames)}")
print(f"Rows generated: {len(rows)}")
print(f"Skipped patient==animate_agent collisions: {collision_skips}")
print(f"Token parity failures: {parity_failures}")
print(f"Saved raw pairs: {OUTPUT_RAW_PAIRS.resolve()}")

Frames available: 203
Frames used (>= 3 per key): 199
Rows generated: 161431
Skipped patient==animate_agent collisions: 11860
Token parity failures: 0
Saved raw pairs: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\raw_pairs.jsonl


## Step 6 - LLM Plausibility Filter

This step scores clean and corrupt prefixes separately in batches, caches scores by UID, and filters pairs with:
- `clean_score >= 4`
- `corrupt_score <= 2`
- `clean_score - corrupt_score >= 2`

Implementation notes:
- Uses `gpt-4o-mini` for pass 1
- Scores clean and corrupt in separate calls
- Supports resume via `phase4_score_cache.jsonl`
- Optional borderline recheck (`corrupt_score == 3`) with `gpt-4o`

In [ ]:
from math import ceil

OUTPUT_FILTERED = Path("../dataset/semantic_meaningful/filtered_pairs.jsonl")
OUTPUT_REJECTED = Path("../dataset/semantic_meaningful/rejected_pairs.jsonl")
OUTPUT_PHASE4_FILTERED = Path("../dataset/semantic_meaningful/phase4_filtered.jsonl")
OUTPUT_PHASE4_REJECTED = Path("../dataset/semantic_meaningful/phase4_rejected.jsonl")
OUTPUT_PHASE4_CACHE = Path("../dataset/semantic_meaningful/phase4_score_cache.jsonl")

PASS1_MODEL = "gpt-4o-mini"
PASS2_MODEL = "gpt-4o"
BATCH_SIZE = 25
RECHECK_BORDERLINE = True
MAX_NEW_UIDS_PER_RUN = 2500

CLEAN_PASS_MIN = 4
CORRUPT_PASS_MIN = 4

assert OUTPUT_RAW_PAIRS.exists(), f"Missing raw pairs file: {OUTPUT_RAW_PAIRS.resolve()}"

raw_pairs = []
with open(OUTPUT_RAW_PAIRS, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_pairs.append(json.loads(line))

raw_by_uid = {row["uid"]: row for row in raw_pairs}

print(f"Raw pairs loaded: {len(raw_pairs):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Pass 1 model: {PASS1_MODEL}")
print(f"Pass 2 model (borderline optional): {PASS2_MODEL}")
print(f"Run cap (new UIDs): {MAX_NEW_UIDS_PER_RUN:,}")
print(f"Score cache path: {OUTPUT_PHASE4_CACHE.resolve()}")

Raw pairs loaded: 161,431
Batch size: 25
Pass 1 model: gpt-4o-mini
Pass 2 model (borderline optional): gpt-4o
Run cap (new UIDs): 2,500
Score cache path: C:\Users\samue\OneDrive\Documenti\Studio\Università\Ricerca\MechInt\grammatical-circuits\animacy-circuit\dataset\semantic_meaningful\phase4_score_cache.jsonl


In [22]:
def load_phase4_cache(cache_path: Path) -> dict[str, dict]:
    cache = {}
    if not cache_path.exists():
        return cache
    with open(cache_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            uid = rec.get("uid")
            if uid:
                cache[uid] = rec
    return cache


def append_phase4_cache(cache_path: Path, records: list[dict]) -> None:
    if not records:
        return
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    with open(cache_path, "a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def build_scoring_prompt(prefixes: list[str]) -> str:
    return (
        "You are a plausibility judge for English sentence prefixes.\n"
        "Rate how plausible each prefix is as the beginning of a real-world event description.\n"
        "Focus on whether the combination of patient and verb is semantically coherent.\n"
        "Ignore the incomplete ending - all sentences end with 'by the'.\n"
        "Score: 1 = impossible, 2 = very odd, 3 = marginal, 4 = plausible, 5 = completely natural.\n"
        "Respond with a JSON array, one object per sentence, in order:\n"
        "[{\"score\": int, \"reason\": \"one sentence\"}, ...]\n\n"
        f"Prefixes:\n{json.dumps(prefixes, ensure_ascii=False)}"
    )


def score_prefix_batch(prefixes: list[str], model_name: str) -> list[dict]:
    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "user",
                "content": build_scoring_prompt(prefixes),
            }
        ],
    )
    payload_text = response.choices[0].message.content
    payload = json.loads(payload_text)

    if isinstance(payload, dict):
        if "scores" in payload and isinstance(payload["scores"], list):
            items = payload["scores"]
        elif "results" in payload and isinstance(payload["results"], list):
            items = payload["results"]
        else:
            # Handle dict-like wrappers with numeric keys.
            items = [payload[k] for k in sorted(payload.keys()) if str(k).isdigit()]
    elif isinstance(payload, list):
        items = payload
    else:
        raise ValueError("Unexpected score payload shape from LLM")

    if len(items) != len(prefixes):
        raise ValueError(f"Score count mismatch: got {len(items)}, expected {len(prefixes)}")

    normalized = []
    for item in items:
        score = int(item.get("score", 0))
        reason = str(item.get("reason", ""))
        score = max(1, min(5, score))
        normalized.append({"score": score, "reason": reason})
    return normalized


def chunks(seq: list, n: int):
    for i in range(0, len(seq), n):
        yield seq[i : i + n]


def is_quota_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return "insufficient_quota" in msg or "exceeded your current quota" in msg


if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is missing. Set it before running Step 6.")

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

phase4_cache = load_phase4_cache(OUTPUT_PHASE4_CACHE)

raw_scored_uids = {uid for uid in raw_by_uid if uid in phase4_cache and "clean_score" in phase4_cache[uid] and "corrupt_score" in phase4_cache[uid]}

missing_uids = [
    uid
    for uid, row in raw_by_uid.items()
    if uid not in phase4_cache
    or "clean_score" not in phase4_cache[uid]
    or "corrupt_score" not in phase4_cache[uid]
]

if MAX_NEW_UIDS_PER_RUN > 0:
    missing_uids = missing_uids[:MAX_NEW_UIDS_PER_RUN]

print(f"Existing phase4 cache entries: {len(phase4_cache):,}")
print(f"Cache entries matching current raw UIDs: {len(raw_scored_uids):,}")
print(f"UIDs needing scoring this run: {len(missing_uids):,}")
print(f"Total clean batches this run: {ceil(len(missing_uids) / BATCH_SIZE) if missing_uids else 0}")

new_records = []
quota_hit = False

for bi, uid_batch in enumerate(chunks(missing_uids, BATCH_SIZE), start=1):
    clean_prefixes = [raw_by_uid[uid]["clean"] for uid in uid_batch]
    corrupt_prefixes = [raw_by_uid[uid]["corrupt"] for uid in uid_batch]

    try:
        clean_scores = score_prefix_batch(clean_prefixes, PASS1_MODEL)
        corrupt_scores = score_prefix_batch(corrupt_prefixes, PASS1_MODEL)
    except Exception as e:
        print(f"Scoring stopped at batch {bi}: {e}")
        if is_quota_error(e):
            quota_hit = True
        break

    for i, uid in enumerate(uid_batch):
        rec = {
            "uid": uid,
            "clean_score": clean_scores[i]["score"],
            "clean_reason": clean_scores[i]["reason"],
            "corrupt_score": corrupt_scores[i]["score"],
            "corrupt_reason": corrupt_scores[i]["reason"],
            "model": PASS1_MODEL,
        }
        phase4_cache[uid] = rec
        new_records.append(rec)

    append_phase4_cache(OUTPUT_PHASE4_CACHE, new_records)
    new_records = []

    if bi % 5 == 0:
        print(f"Processed {bi} clean/corrupt batch pairs")

print(f"Phase 4 cache entries after pass 1: {len(phase4_cache):,}")

if RECHECK_BORDERLINE and not quota_hit:
    borderline_uids = [
        uid
        for uid, rec in phase4_cache.items()
        if uid in raw_by_uid and rec.get("corrupt_score") == 3
]
    if MAX_NEW_UIDS_PER_RUN > 0:
        borderline_uids = borderline_uids[: max(250, min(MAX_NEW_UIDS_PER_RUN, 1500))]

    print(f"Borderline corrupt=3 pairs for optional recheck: {len(borderline_uids):,}")

    for bi, uid_batch in enumerate(chunks(borderline_uids, BATCH_SIZE), start=1):
        corrupt_prefixes = [raw_by_uid[uid]["corrupt"] for uid in uid_batch]

        try:
            rescored = score_prefix_batch(corrupt_prefixes, PASS2_MODEL)
        except Exception as e:
            print(f"Borderline recheck stopped at batch {bi}: {e}")
            break

        overwrite_records = []
        for i, uid in enumerate(uid_batch):
            rec = phase4_cache[uid]
            rec["corrupt_score"] = rescored[i]["score"]
            rec["corrupt_reason"] = rescored[i]["reason"]
            rec["model"] = PASS2_MODEL
            overwrite_records.append(rec)

        append_phase4_cache(OUTPUT_PHASE4_CACHE, overwrite_records)

        if bi % 5 == 0:
            print(f"Rechecked {bi} borderline batches")

elif RECHECK_BORDERLINE and quota_hit:
    print("Skipping borderline recheck due to quota/rate-limit stop in pass 1.")

print("Step 6 scoring pass completed for this run.")

Existing phase4 cache entries: 12,180
Cache entries matching current raw UIDs: 0
UIDs needing scoring this run: 2,500
Total clean batches this run: 100
Scoring stopped at batch 1: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Phase 4 cache entries after pass 1: 12,180
Skipping borderline recheck due to quota/rate-limit stop in pass 1.
Step 6 scoring pass completed for this run.


In [ ]:
filtered_rows = []
rejected_rows = []
rejection_counts = defaultdict(int)

scored_rows_count = 0
for row in raw_pairs:
    uid = row["uid"]
    rec = phase4_cache.get(uid)
    if not rec:
        continue
    if "clean_score" not in rec or "corrupt_score" not in rec:
        continue

    scored_rows_count += 1
    clean_score = int(rec.get("clean_score", 0))
    corrupt_score = int(rec.get("corrupt_score", 0))
    gap = clean_score - corrupt_score

    reasons = []
    if clean_score < CLEAN_PASS_MIN:     # clean >= 4
        reasons.append("clean too low")
    if corrupt_score < CORRUPT_PASS_MIN: # corrupt >= 4 (same threshold)
        reasons.append("corrupt too low")
    # Remove the gap filter entirely

    enriched = {
        **row,
        "clean_score": clean_score,
        "clean_reason": rec.get("clean_reason", ""),
        "corrupt_score": corrupt_score,
        "corrupt_reason": rec.get("corrupt_reason", ""),
        "score_model": rec.get("model", PASS1_MODEL),
    }

    if reasons:
        enriched["rejection_reasons"] = reasons
        rejected_rows.append(enriched)
        for reason in reasons:
            rejection_counts[reason] += 1
    else:
        filtered_rows.append(enriched)

print(f"Scored rows available: {scored_rows_count:,} / {len(raw_pairs):,}")
print(f"Kept rows: {len(filtered_rows):,}")
print(f"Rejected rows: {len(rejected_rows):,}")

if scored_rows_count == 0:
    print("No scored rows for current raw_pairs UID set. Skipping file writes to avoid empty-output overwrite.")
else:
    for out_path, data in [
        (OUTPUT_FILTERED, filtered_rows),
        (OUTPUT_REJECTED, rejected_rows),
        (OUTPUT_PHASE4_FILTERED, filtered_rows),
        (OUTPUT_PHASE4_REJECTED, rejected_rows),
    ]:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "w", encoding="utf-8") as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("\nRejection reasons:")
    for reason, count in sorted(rejection_counts.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"  {reason:<17} {count:,}")

    print(f"\nSaved filtered pairs: {OUTPUT_FILTERED.resolve()}")
    print(f"Saved rejected pairs: {OUTPUT_REJECTED.resolve()}")
    print(f"Compatibility export filtered: {OUTPUT_PHASE4_FILTERED.resolve()}")
    print(f"Compatibility export rejected: {OUTPUT_PHASE4_REJECTED.resolve()}")

Scored rows available: 0 / 161,431
Kept rows: 0
Rejected rows: 0
No scored rows for current raw_pairs UID set. Skipping file writes to avoid empty-output overwrite.
